# vclone: a talking video in the person's own voice

Upload a **~30 second video** of someone talking to the camera and type a **script**. You get back a video of
them saying the script **in their own cloned voice, lip-synced**, with their real head movement and expressions.

1. **Runtime → Change runtime type → T4 GPU** (free tier works).
2. Run the cells from top to bottom.

Only use videos of people who agreed to be cloned. Every output is tagged as AI-generated in its metadata.

In [ ]:
REPO_URL = "https://github.com/dhanuvagman006/VedioFaceClone.git"  # your GitHub repo
KEEP_MODELS_ON_DRIVE = False  # True: keep the ~15 GB of models in Google Drive so later sessions skip the download

## 1. Get the latest code (and check the GPU)

In [ ]:
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if os.path.exists("/content/vclone"):
    !git -C /content/vclone pull -q
else:
    !git clone -q {REPO_URL} /content/vclone
%cd /content/vclone
!git log --oneline -1
if KEEP_MODELS_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/vclone-models", exist_ok=True)
    if not os.path.islink("models"):
        !rm -rf models && ln -s /content/drive/MyDrive/vclone-models models

## 2. Install and download the models (first run ~5-10 minutes)

In [ ]:
!bash setup.sh --system 2>&1 | grep -v "^\s*$" | tail -n 8

## 3. Your script and video

Tip for the most accurate voice: in the video, have the person read the text printed by `./speak.sh --script`
(about 10 s). The tool recognizes it and uses its exact words to learn the voice.
Good video: face visible and front-on, good light, quiet room, 25-60 seconds. Holding the phone at arm's
length (head and shoulders in view) gives a sharper mouth than an extreme close-up.

In [ ]:
SCRIPT = """Hi everyone! This whole video was made from a thirty second recording and this text.
Pretty wild, right?"""
QUALITY = "high"  # fast = 1 take, high = best of 3 (default), max = best of 5

In [ ]:
from google.colab import files
print("Upload the video of the person (mp4 / mov / webm):")
VIDEO = next(iter(files.upload()))

## 4. Make the video

In [ ]:
with open("script.txt", "w", encoding="utf-8") as f:
    f.write(SCRIPT)
!./speak.sh "{VIDEO}" -f script.txt -o outputs/result.mp4 --quality {QUALITY}

In [ ]:
from IPython.display import Video, display
display(Video("outputs/result.mp4", embed=True, width=480))
files.download("outputs/result.mp4")

**Next scripts are faster:** change `SCRIPT` and re-run step 4. The voice and face analysis of the same video
are cached, so only the new speech and lip sync are generated.

On this T4 the bigger **1.7B voice model** is used automatically (closer to the real voice), and the lip-synced
mouth is sharpened with **GFPGAN** face restoration. Options you can add to the `./speak.sh` line:
`--restore 1.0` (sharper mouth) or `--restore 0.5` (more natural, softer), `--restore 0` (off),
`--language french` (10 languages), `--quality max`, `--seed 42` (reproducible), `-V` (show scores).